Importing the necessary libraries


In [ ]:
# Libraries to help with reading and manipulating data
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from great_tables import GT, loc, style

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# to restrict the float value to 3 decimal places
pd.set_option('display.float_format', lambda x: '%.3f' % x)

Importing the dataset

In [ ]:
# Load the raw validation workbook.
# Resolves the repo-relative de-identified copy regardless of the working
# directory, and falls back to the author's original shared-drive location.
def _find_workbook(fname="merged_llm_summary_validation_datasheet.xlsx"):
    here = Path.cwd()
    for base in [here, *here.parents]:
        candidate = base / "data" / "raw" / fname
        if candidate.exists():
            return candidate
    # Fallback: original shared G: drive (author's research environment)
    return Path("G:/Documents/Research/Projects/moo/llm_summary/data/raw") / fname

data_path = _find_workbook()
data = pd.read_excel(data_path)
print(f"Loaded {data_path}  \u2192  {data.shape[0]} rows \u00d7 {data.shape[1]} cols")

Understand the data

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:

data['lesion_size_status_source'].unique()

In [ ]:
data['lesion_size_status_source'].value_counts()

In [ ]:
data['lesion_location_status_human'].value_counts()

Handle Missing Values

In [ ]:
# Convert common placeholder strings to actual missing values across the DataFrame
missing_placeholders = ['na', 'NA', 'N/A']
data.replace(to_replace=missing_placeholders, value=np.nan, inplace=True)

In [ ]:
# Data not present in source documents, hence missing in both human and AI annotations
data.isnull().sum()

In [ ]:
# Drop rows absent from the source documents: those null across ALL human and AI
# annotation columns. (A blanket dropna() would drop any row with any NaN and
# collapse 200 cases to ~20; per-element accuracy below scores only source-present cases.)
human_ai_cols = [c for c in data.columns
                 if c.endswith('_status_human') or c.endswith('_status_ai')]
data = data.dropna(subset=human_ai_cols, how='all')

Encode variables 

Accuracy metric calculations

In [ ]:
# Proportion of lesion size annotations by humans that is accurate
source_presence_lesion_size = (data['lesion_size_status_source'] == 1).sum()
human_lesion_size_annotation = ((data['lesion_size_status_source'] == 1) & (data['lesion_size_status_human'] == 1)).sum()
lesion_size_human_accuracy = human_lesion_size_annotation / source_presence_lesion_size

In [ ]:

# Proportion of lesions size annotations by ai that is accurate
source_presence_lesion_size = (data['lesion_size_status_source'] == 1).sum()
ai_lesion_size_annotation = ((data['lesion_size_status_source'] == 1) & (data['lesion_size_status_ai'] == 1)).sum()
lesion_size_ai_accuracy = ai_lesion_size_annotation / source_presence_lesion_size

In [ ]:
# Proportion of lesion laterality annotations by humans that is accurate
source_presence_lesion_laterality  = (data['laterality_status_source'] == 1).sum()
human_lesion_laterality_annotation = ((data['laterality_status_source'] == 1) & (data['laterality_status_human'] == 1)).sum()
lesion_laterality_human_accuracy = human_lesion_laterality_annotation / source_presence_lesion_laterality

In [ ]:
# Proportion of lesion laterality annotations by ai that is accurate
source_presence_lesion_laterality  = (data['laterality_status_source'] == 1).sum()
ai_lesion_laterality_annotation = ((data['laterality_status_source'] == 1) & (data['laterality_status_ai'] == 1)).sum()
lesion_laterality_ai_accuracy = ai_lesion_laterality_annotation / source_presence_lesion_laterality

In [ ]:
# Proportion of lesion location annotations by humans that is accurate

source_presence_lesion_location = (data['lesion_location_status_source'] == 1).sum()
human_lesion_annotation = ((data['lesion_location_status_source'] == 1) & (data['lesion_location_status_human'] == 1)).sum()
lesion_location_human_accuracy = human_lesion_annotation / source_presence_lesion_location

In [ ]:
# Proportion of lesion location annotations by AI that is accurate

source_presence_lesion_location = (data['lesion_location_status_source'] == 1).sum()
ai_lesion_annotation = ((data['lesion_location_status_source'] == 1) & (data['lesion_location_status_ai'] == 1)).sum()
lesion_location_ai_accuracy = ai_lesion_annotation / source_presence_lesion_location

In [ ]:
# Proportion of calcifications and asymmetry annotations by humans that is accurate
source_presence_calcifications_asymmetry = (data['calcifications_asymmetry_status_source'] == 1).sum()
human_calcifications_asymmetry_annotation = ((data['calcifications_asymmetry_status_source'] == 1) & (data['calcifications_asymmetry_status_human'] == 1)).sum()
calcifications_asymmetry_human_accuracy = human_calcifications_asymmetry_annotation / source_presence_calcifications_asymmetry

In [ ]:
# Proportion of calcifications and asymmetry annotations by ai that is accurate
source_presence_calcifications_asymmetry = (data['calcifications_asymmetry_status_source'] == 1).sum()
ai_calcifications_asymmetry_annotation = ((data['calcifications_asymmetry_status_source'] == 1) & (data['calcifications_asymmetry_status_ai'] == 1)).sum()
calcifications_asymmetry_ai_accuracy = ai_calcifications_asymmetry_annotation / source_presence_calcifications_asymmetry

In [ ]:
# Proportion of enhancement pattern annotations by humans that is accurate
source_presence_enhancement_pattern = (data['additional_enhancement_mri_status_source'] == 1).sum()
human_enhancement_pattern_annotation = ((data['additional_enhancement_mri_status_source'] == 1) & (data['additional_enhancement_mri_status_human'] == 1)).sum()
enhancement_pattern_human_accuracy = human_enhancement_pattern_annotation / source_presence_enhancement_pattern     

In [ ]:
# Proportion of enhancement pattern annotations by ai that is accurate
source_presence_enhancement_pattern = (data['additional_enhancement_mri_status_source'] == 1).sum()
ai_enhancement_pattern_annotation = ((data['additional_enhancement_mri_status_source'] == 1) & (data['additional_enhancement_mri_status_ai'] == 1)).sum()
enhancement_pattern_ai_accuracy = ai_enhancement_pattern_annotation / source_presence_enhancement_pattern   

In [ ]:
# Proportion of enhancement extent annotations by humans that is accurate
source_presence_enhancement_extent = (data['extent_status_source'] == 1).sum()
human_enhancement_extent_annotation = ((data['extent_status_source'] == 1) & (data['extent_status_human'] == 1)).sum()
enhancement_extent_human_accuracy = human_enhancement_extent_annotation / source_presence_enhancement_extent        

In [ ]:
# Proportion of enhancement extent annotations by ai that is accurate
source_presence_enhancement_extent = (data['extent_status_source'] == 1).sum()
ai_enhancement_extent_annotation = ((data['extent_status_source'] == 1) & (data['extent_status_ai'] == 1)).sum()
enhancement_extent_ai_accuracy = ai_enhancement_extent_annotation / source_presence_enhancement_extent      

In [ ]:
# Proportion of clip placement annotations by humans that is accurate
source_presence_clip_placement = (data['accurate_clip_placement_status_source'] == 1).sum()
human_clip_placement_annotation = ((data['accurate_clip_placement_status_source'] == 1) & (data['accurate_clip_placement_status_human'] == 1)).sum()
clip_placement_human_accuracy = human_clip_placement_annotation / source_presence_clip_placement

In [ ]:
# Proportion of clip placement annotation by ai that is accurate
source_presence_clip_placement = (data['accurate_clip_placement_status_source'] == 1).sum()
ai_clip_placement_annotation = ((data['accurate_clip_placement_status_source'] == 1) & (data['accurate_clip_placement_status_ai'] == 1)).sum()
clip_placement_ai_accuracy = ai_clip_placement_annotation / source_presence_clip_placement

In [ ]:
# Proportion of workup recommendations annotated by humans that is accurate
source_presence_workup_recommendation = (data['workup_recommendation_status_source'] == 1).sum()
human_workup_recommendation_annotation = ((data['workup_recommendation_status_source'] == 1) & (data['workup_recommendation_status_human'] == 1)).sum()
workup_recommendation_human_accuracy = human_workup_recommendation_annotation / source_presence_workup_recommendation

In [ ]:
# Proportion of workup recommendations annotated by ai that is accurate
source_presence_workup_recommendation = (data['workup_recommendation_status_source'] == 1).sum()
ai_workup_recommendation_annotation = ((data['workup_recommendation_status_source'] == 1) & (data['workup_recommendation_status_ai'] == 1)).sum()
workup_recommendation_ai_accuracy = ai_workup_recommendation_annotation / source_presence_workup_recommendation

In [ ]:
# Proportion of lymph node status annotations by humans that is accurate
source_presence_lymph_node_status = (data['Lymph node_status_source'] == 1).sum()
human_lymph_node_status_annotation = ((data['Lymph node_status_source'] == 1) & (data['Lymph node_status_human'] == 1)).sum()
lymph_node_status_human_accuracy = human_lymph_node_status_annotation / source_presence_lymph_node_status

In [ ]:
# Proportion of lymph node status annotations by ai that is accurate
source_presence_lymph_node_status = (data['Lymph node_status_source'] == 1).sum()
ai_lymph_node_status_annotation = ((data['Lymph node_status_source'] == 1) & (data['Lymph node_status_ai'] == 1)).sum()
lymph_node_status_ai_accuracy = ai_lymph_node_status_annotation / source_presence_lymph_node_status

In [ ]:
# Proportion of breast oncologic timelines by humans that is accurate
source_presence_breast_oncologic_timeline = (data['chronology_preserved_status_source'] == 1).sum()
human_breast_oncologic_timeline_annotation = ((data['chronology_preserved_status_source'] == 1) & (data['chronology_preserved_status_human'] == 1)).sum()
breast_oncologic_timeline_human_accuracy = human_breast_oncologic_timeline_annotation / source_presence_breast_oncologic_timeline   

In [ ]:
# Proportion of breast oncologic timelines by ai that is accurate   
source_presence_breast_oncologic_timeline = (data['chronology_preserved_status_source'] == 1).sum() 
ai_breast_oncologic_timeline_annotation = ((data['chronology_preserved_status_source'] == 1) & (data['chronology_preserved_status_ai'] == 1)).sum()
breast_oncologic_timeline_ai_accuracy = ai_breast_oncologic_timeline_annotation / source_presence_breast_oncologic_timeline 

In [ ]:
# Proportion of biopsy method annotations by humans that is accurate
source_presence_biopsy_method = (data['biopsy_method_status_source'] == 1).sum()
human_biopsy_method_annotation = ((data['biopsy_method_status_source'] == 1) & (data['biopsy_method_status_human'] == 1)).sum()
biopsy_method_human_accuracy = human_biopsy_method_annotation / source_presence_biopsy_method

In [ ]:
# Proportion of biopsy method annotations by ai that is accurate
source_presence_biopsy_method = (data['biopsy_method_status_source'] == 1).sum()
ai_biopsy_method_annotation = ((data['biopsy_method_status_source'] == 1) & (data['biopsy_method_status_ai'] == 1)).sum()
biopsy_method_ai_accuracy = ai_biopsy_method_annotation / source_presence_biopsy_method 

In [ ]:
# Proportion of invasive component size annotations by humans that is accurate
source_presence_invasive_component = (data['invasive_component_size_pathology_status_source'] == 1).sum()
human_invasive_component_annotation = ((data['invasive_component_size_pathology_status_source'] == 1) & (data['invasive_component_size_pathology_status_human'] == 1)).sum()
invasive_component_human_accuracy = human_invasive_component_annotation / source_presence_invasive_component

In [ ]:
# Proportion of invasive component size annotations by ai that is accurate
source_presence_invasive_component = (data['invasive_component_size_pathology_status_source'] == 1).sum()
ai_invasive_component_annotation = ((data['invasive_component_size_pathology_status_source'] == 1) & (data['invasive_component_size_pathology_status_ai'] == 1)).sum()
invasive_component_ai_accuracy = ai_invasive_component_annotation / source_presence_invasive_component  

In [ ]:
# Proportion of histologic diagnosis annotations by humans that is accurate
source_presence_histologic_diagnosis = (data['histologic_diagnosis_status_source'] == 1).sum()
human_histologic_diagnosis_annotation = ((data['histologic_diagnosis_status_source'] == 1) & (data['histologic_diagnosis_status_human'] == 1)).sum()
histologic_diagnosis_human_accuracy = human_histologic_diagnosis_annotation / source_presence_histologic_diagnosis  

In [ ]:
# Proportion of histologic diagnosis annotations by ai that is accurate
source_presence_histologic_diagnosis = (data['histologic_diagnosis_status_source'] == 1).sum()  
ai_histologic_diagnosis_annotation = ((data['histologic_diagnosis_status_source'] == 1) & (data['histologic_diagnosis_status_ai'] == 1)).sum()
histologic_diagnosis_ai_accuracy = ai_histologic_diagnosis_annotation / source_presence_histologic_diagnosis    

In [ ]:
# Proportion of receptor status annotations by humans that is accurate
source_presence_receptor_status = (data['receptor_status_source'] == 1).sum()
human_receptor_status_annotation = ((data['receptor_status_source'] == 1) & (data['receptor_status_human'] == 1)).sum()
receptor_status_human_accuracy = human_receptor_status_annotation / source_presence_receptor_status 

In [ ]:
# Proportion of receptor status annotation by ai that is accurate
source_presence_receptor_status = (data['receptor_status_source'] == 1).sum()
ai_receptor_status_annotation = ((data['receptor_status_source'] == 1) & (data['receptor_status_ai'] == 1)).sum()
receptor_status_ai_accuracy = ai_receptor_status_annotation / source_presence_receptor_status   

Accuracy metric summary for human vs ai 

In [ ]:
# Human and AI element-level accuracies summarized in one table
data_human_accuracy_element_level = {
    'Lesion size': lesion_size_human_accuracy,
    'Lesion laterality': lesion_laterality_human_accuracy,
    'Lesion location': lesion_location_human_accuracy,
    'Calcifications and asymmetry': calcifications_asymmetry_human_accuracy,
    'Enhancement pattern': enhancement_pattern_human_accuracy,
    'Enhancement extent': enhancement_extent_human_accuracy,
    'Clip placement and displacement': clip_placement_human_accuracy,
    'Workup recommendations': workup_recommendation_human_accuracy,
    'Lymph node status': lymph_node_status_human_accuracy,
    'Breast oncologic timeline': breast_oncologic_timeline_human_accuracy,
    'Biopsy method': biopsy_method_human_accuracy,
    'Invasive component': invasive_component_human_accuracy,
    'Histologic diagnosis': histologic_diagnosis_human_accuracy,
    'Receptor status': receptor_status_human_accuracy
}

data_ai_accuracy_element_level = {
    'Lesion size': lesion_size_ai_accuracy,
    'Lesion laterality': lesion_laterality_ai_accuracy,
    'Lesion location': lesion_location_ai_accuracy,
    'Calcifications and asymmetry': calcifications_asymmetry_ai_accuracy,
    'Enhancement pattern': enhancement_pattern_ai_accuracy,
    'Enhancement extent': enhancement_extent_ai_accuracy,
    'Clip placement and displacement': clip_placement_ai_accuracy,
    'Workup recommendations': workup_recommendation_ai_accuracy,
    'Lymph node status': lymph_node_status_ai_accuracy,
    'Breast oncologic timeline': breast_oncologic_timeline_ai_accuracy,
    'Biopsy method': biopsy_method_ai_accuracy,
    'Invasive component': invasive_component_ai_accuracy,
    'Histologic diagnosis': histologic_diagnosis_ai_accuracy,
    'Receptor status': receptor_status_ai_accuracy
}

df_human_ai_element_accuracy = (
    pd.DataFrame.from_dict(
        data_human_accuracy_element_level, orient='index', columns=['Human Accuracy']
    )
    .join(
        pd.DataFrame.from_dict(
            data_ai_accuracy_element_level, orient='index', columns=['AI Accuracy']
        )
    )
    .rename_axis('Element')
    .reset_index()
 )

df_human_ai_element_accuracy

In [ ]:
# Hypothesis tests and confidence intervals (95%) for human vs. AI element accuracy
# using McNemar's test and adjusted Wald (Agresti-Coull) CIs. Accuracy and its CI are
# scored only over source-present cases, so successes never exceed the denominator.
from statsmodels.stats.contingency_tables import mcnemar

def adjusted_wald_ci(successes, n, alpha=0.05):
    # Agresti-Coull (adjusted Wald) interval for a binomial proportion
    if n == 0:
        return (float('nan'), float('nan'))
    z = 1.96  # 95% CI
    n_adj = n + z**2
    p_adj = (successes + 0.5 * z**2) / n_adj
    se_adj = (p_adj * (1 - p_adj) / n_adj) ** 0.5
    lower = p_adj - z * se_adj
    upper = p_adj + z * se_adj
    return max(0, lower), min(1, upper)

# Element -> (human, ai, source) actual columns (the source schema is not uniform).
element_columns = {
    'Lesion size': ('lesion_size_status_human', 'lesion_size_status_ai', 'lesion_size_status_source'),
    'Lesion laterality': ('laterality_status_human', 'laterality_status_ai', 'laterality_status_source'),
    'Lesion location': ('lesion_location_status_human', 'lesion_location_status_ai', 'lesion_location_status_source'),
    'Calcifications and asymmetry': ('calcifications_asymmetry_status_human', 'calcifications_asymmetry_status_ai', 'calcifications_asymmetry_status_source'),
    'Enhancement pattern': ('additional_enhancement_mri_status_human', 'additional_enhancement_mri_status_ai', 'additional_enhancement_mri_status_source'),
    'Enhancement extent': ('extent_status_human', 'extent_status_ai', 'extent_status_source'),
    'Clip placement and displacement': ('accurate_clip_placement_status_human', 'accurate_clip_placement_status_ai', 'accurate_clip_placement_status_source'),
    'Workup recommendations': ('workup_recommendation_status_human', 'workup_recommendation_status_ai', 'workup_recommendation_status_source'),
    'Lymph node status': ('Lymph node_status_human', 'Lymph node_status_ai', 'Lymph node_status_source'),
    'Breast oncologic timeline': ('chronology_preserved_status_human', 'chronology_preserved_status_ai', 'chronology_preserved_status_source'),
    'Biopsy method': ('biopsy_method_status_human', 'biopsy_method_status_ai', 'biopsy_method_status_source'),
    'Invasive component': ('invasive_component_size_pathology_status_human', 'invasive_component_size_pathology_status_ai', 'invasive_component_size_pathology_status_source'),
    'Histologic diagnosis': ('histologic_diagnosis_status_human', 'histologic_diagnosis_status_ai', 'histologic_diagnosis_status_source'),
    'Receptor status': ('receptor_status_human', 'receptor_status_ai', 'receptor_status_source'),
}

records = []
element_keys = [
    ('Lesion size', human_lesion_size_annotation, ai_lesion_size_annotation, source_presence_lesion_size),
    ('Lesion laterality', human_lesion_laterality_annotation, ai_lesion_laterality_annotation, source_presence_lesion_laterality),
    ('Lesion location', human_lesion_annotation, ai_lesion_annotation, source_presence_lesion_location),
    ('Calcifications and asymmetry', human_calcifications_asymmetry_annotation, ai_calcifications_asymmetry_annotation, source_presence_calcifications_asymmetry),
    ('Enhancement pattern', human_enhancement_pattern_annotation, ai_enhancement_pattern_annotation, source_presence_enhancement_pattern),
    ('Enhancement extent', human_enhancement_extent_annotation, ai_enhancement_extent_annotation, source_presence_enhancement_extent),
    ('Clip placement and displacement', human_clip_placement_annotation, ai_clip_placement_annotation, source_presence_clip_placement),
    ('Workup recommendations', human_workup_recommendation_annotation, ai_workup_recommendation_annotation, source_presence_workup_recommendation),
    ('Lymph node status', human_lymph_node_status_annotation, ai_lymph_node_status_annotation, source_presence_lymph_node_status),
    ('Breast oncologic timeline', human_breast_oncologic_timeline_annotation, ai_breast_oncologic_timeline_annotation, source_presence_breast_oncologic_timeline),
    ('Biopsy method', human_biopsy_method_annotation, ai_biopsy_method_annotation, source_presence_biopsy_method),
    ('Invasive component', human_invasive_component_annotation, ai_invasive_component_annotation, source_presence_invasive_component),
    ('Histologic diagnosis', human_histologic_diagnosis_annotation, ai_histologic_diagnosis_annotation, source_presence_histologic_diagnosis),
    ('Receptor status', human_receptor_status_annotation, ai_receptor_status_annotation, source_presence_receptor_status),
]
for element, human_success, ai_success, total in element_keys:
    if total == 0:
        continue
    human_ci = adjusted_wald_ci(human_success, total)
    ai_ci = adjusted_wald_ci(ai_success, total)
    human_col, ai_col, source_col = element_columns[element]
    # McNemar over source-present cases where both annotations are observed
    mask = (data[source_col] == 1) & data[[human_col, ai_col]].notnull().all(axis=1)
    if mask.sum() == 0:
        p_value = float('nan')
    else:
        human_bin = (data.loc[mask, human_col] == 1).astype(int)
        ai_bin = (data.loc[mask, ai_col] == 1).astype(int)
        table = pd.crosstab(human_bin, ai_bin).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
        discordant = int(table.iloc[0, 1]) + int(table.iloc[1, 0])
        if discordant == 0:
            p_value = 1.0  # no discordant pairs -> no evidence of a difference
        else:
            # exact binomial test when discordant pairs are few, chi-square otherwise
            result = mcnemar(table, exact=discordant < 25)
            p_value = result.pvalue
    records.append({
        'Element': element,
        'Human Accuracy': human_success / total,
        'AI Accuracy': ai_success / total,
        'n (source-present)': int(total),
        'Human 95% CI': f"({human_ci[0]:.3f}, {human_ci[1]:.3f})",
        'AI 95% CI': f"({ai_ci[0]:.3f}, {ai_ci[1]:.3f})",
        'Human CI Low': human_ci[0],
        'Human CI High': human_ci[1],
        'AI CI Low': ai_ci[0],
        'AI CI High': ai_ci[1],
        'p_value (McNemar)': p_value,
    })

df_accuracy_tests = pd.DataFrame(records)
df_accuracy_tests

In [ ]:
# Format df_accuracy_tests as a Great Tables object with confidence intervals shown beneath each percentage

def render_p_value(value):
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        return "-"
    if np.isnan(numeric_value):
        return "-"
    return "<0.001" if numeric_value < 1e-3 else f"{numeric_value:.3f}"


def format_accuracy_with_ci(value, ci_text):
    if pd.isna(value):
        return "n/a"
    return f"{value:.1f}%\n{ci_text}"


def apply_ci_formatter(gt_object, column, ci_values):
    for row_idx, ci_text in enumerate(ci_values):
        gt_object = gt_object.fmt(
            lambda cell_value, ci=ci_text: format_accuracy_with_ci(cell_value, ci),
            columns=column,
            rows=[row_idx],
        )
    return gt_object


human_ci_labels = df_accuracy_tests['Human 95% CI'].tolist()
ai_ci_labels = df_accuracy_tests['AI 95% CI'].tolist()


df_accuracy_table = (
    df_accuracy_tests[[
        'Element',
        'Human Accuracy',
        'AI Accuracy',
        'p_value (McNemar)',
    ]]
    .rename(columns={
        'Element': 'element',
        'Human Accuracy': 'human_accuracy',
        'AI Accuracy': 'ai_accuracy',
        'p_value (McNemar)': 'p_value',
    })
    .assign(
        human_accuracy_pct=lambda df_: df_['human_accuracy'] * 100,
        ai_accuracy_pct=lambda df_: df_['ai_accuracy'] * 100,
        accuracy_gap=lambda df_: (df_['human_accuracy'] - df_['ai_accuracy']) * 100,
    )
    [['element', 'human_accuracy_pct', 'ai_accuracy_pct', 'accuracy_gap', 'p_value']]
)


gap_span = max(1.0, float(df_accuracy_table['accuracy_gap'].abs().max()))
gap_domain = [-gap_span, gap_span]


gt_accuracy_table = (
    GT(df_accuracy_table)
    .tab_header(
        title="Element-level accuracy: Human vs. AI",
        subtitle="Percent accuracy with Agresti-Coull (adjusted Wald) 95% CIs; p-values from McNemar's test.",
    )
    .cols_label(
        element="Element",
        human_accuracy_pct="Human Accuracy (%)",
        ai_accuracy_pct="AI Accuracy (%)",
        accuracy_gap="Δ (Human − AI)",
        p_value="p-value",
    )
    .fmt_number(columns=['accuracy_gap'], decimals=1)
    .fmt(render_p_value, columns='p_value')
    .data_color(columns='human_accuracy_pct', palette=['#e0f2fe', '#0f172a'], domain=[50, 100])
    .data_color(columns='ai_accuracy_pct', palette=['#fef2f2', '#991b1b'], domain=[50, 100])
    .data_color(columns='accuracy_gap', palette=['#d1fae5', '#047857'], domain=gap_domain)
    .cols_align(align='center', columns=['human_accuracy_pct', 'ai_accuracy_pct', 'accuracy_gap', 'p_value'])
    .opt_row_striping()
    .opt_table_font(font='Segoe UI')
    .tab_options(
        container_width='100%',
        table_width='100%',
        table_layout='auto',
        heading_padding='14px',
        column_labels_padding='12px',
        column_labels_font_size='13px',
    )
    .tab_source_note("Agresti-Coull (adjusted Wald) intervals; paired comparison uses McNemar's test.")
)


gt_accuracy_table = apply_ci_formatter(
    gt_accuracy_table,
    column='human_accuracy_pct',
    ci_values=human_ci_labels,
)


gt_accuracy_table = apply_ci_formatter(
    gt_accuracy_table,
    column='ai_accuracy_pct',
    ci_values=ai_ci_labels,
)


gt_accuracy_table = gt_accuracy_table.tab_style(
    style=style.text(whitespace='pre-line'),
    locations=loc.body(columns=['human_accuracy_pct', 'ai_accuracy_pct']),
)


gt_accuracy_table

In [ ]:
# Side-by-side bar plot comparing human and AI accuracies across elements with 95% CIs
element_order = df_accuracy_tests['Element'].tolist()
plot_records = []
for _, row in df_accuracy_tests.iterrows():
    plot_records.extend([
        {
            'Element': row['Element'],
            'Annotator': 'Human',
            'Accuracy_pct': row['Human Accuracy'] * 100,
            'ci_low_pct': row['Human CI Low'] * 100,
            'ci_high_pct': row['Human CI High'] * 100,
        },
        {
            'Element': row['Element'],
            'Annotator': 'AI',
            'Accuracy_pct': row['AI Accuracy'] * 100,
            'ci_low_pct': row['AI CI Low'] * 100,
            'ci_high_pct': row['AI CI High'] * 100,
        },
    ])
plot_df = pd.DataFrame(plot_records)
plot_df['Element'] = pd.Categorical(plot_df['Element'], categories=element_order, ordered=True)
plot_df['Annotator'] = pd.Categorical(plot_df['Annotator'], categories=['Human', 'AI'], ordered=True)
plot_df_sorted = plot_df.sort_values(['Annotator', 'Element']).reset_index(drop=True)
max_ci_pct = plot_df['ci_high_pct'].max()
fig, ax = plt.subplots(figsize=(12.5, 8.5))
sns.barplot(
    data=plot_df,
    x='Element',
    y='Accuracy_pct',
    hue='Annotator',
    palette=['#0f172a', '#991b1b'],
    edgecolor='black',
    linewidth=0.8,
    width=0.75,
    ax=ax,
    order=element_order,
    hue_order=['Human', 'AI'],
    errorbar=None,
 )
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('Radiologic or Pathologic Element')
y_max = max(110, max_ci_pct + 10)
ax.set_ylim(0, y_max)
ax.set_yticks(np.arange(0, 101, 10))
ax.margins(y=0.06)
ax.set_title('Accuracy (%) of AI vs Human Summaries per Element', pad=40)
plt.xticks(rotation=35, ha='right')
for patch, record in zip(ax.patches, plot_df_sorted.itertuples(index=False)):
    center = patch.get_x() + patch.get_width() / 2
    err_low = max(record.Accuracy_pct - record.ci_low_pct, 0)
    err_high = max(record.ci_high_pct - record.Accuracy_pct, 0)
    ax.errorbar(
        center,
        record.Accuracy_pct,
        yerr=[[err_low], [err_high]],
        color='black',
        capsize=4,
        linewidth=1,
        fmt='none',
        zorder=5,
    )
legend = ax.legend(
    title='Annotator',
    frameon=False,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
    ncol=1,
 )
fig.tight_layout(rect=[0, 0.03, 0.86, 0.95])
plt.show()

Accuracy metric stratified by radiologic vs pathologic elements

Accuracy metric stratified by case complexity

## Export outputs

Save the element-level summary table and comparison figure to `outputs/` for reporting.

In [ ]:
# Export the summary table and figure to outputs/ (aggregate statistics only; no PHI)
from pathlib import Path

def _repo_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    return Path.cwd()

out_dir = _repo_root() / "outputs"
out_dir.mkdir(exist_ok=True)

# Element-level summary (accuracies, CIs, n, McNemar p-values) as CSV
df_accuracy_tests.to_csv(out_dir / "element_accuracy_summary.csv", index=False)

# great_tables summary: HTML always; PNG via gtsave (lightweight nokap, no browser)
(out_dir / "summary_table.html").write_text(gt_accuracy_table.as_raw_html(), encoding="utf-8")
try:
    gt_accuracy_table.gtsave(str(out_dir / "summary_table.png"))
    png_status = "summary_table.png"
except Exception as exc:
    png_status = f"(PNG skipped: {type(exc).__name__})"

# Grouped bar chart with CI error bars
fig.savefig(out_dir / "accuracy_barchart.png", dpi=200, bbox_inches="tight")

print("Exports written to", out_dir)
for name in ("element_accuracy_summary.csv", "summary_table.html", "accuracy_barchart.png"):
    print("  -", name)
print("  - summary_table.png ->", png_status)